# Figure 6I — Modularity
Progressively project paired NK landscapes onto modules $(0,1)$ and $(2,3)$ while preserving each landscape's mean and variance.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from scripts.generate_figure6_i_modularity_raw_data import (
    K_VALUES, LAMBDA_VALUES, MODULES, N_SITES, NUM_ALLELES,
    NUM_DECAY_STEPS, NUM_LANDSCAPES, NUM_LOCAL_STARTS,
    RANDOM_SEED, TOTAL_MUTATION_RATE, modular_landscape,
)
from slide.direvo_functions import get_single_decay_rate
from slide.ruggedness_functions import get_dirichlet_metric
from slide.utils import (
    FIGURE_LABEL_SIZE, FIGURE_LEGEND_SIZE, FIGURE_TICK_SIZE,
    FIGURE_TITLE_SIZE, PANEL_LETTER_SIZE, get_figures_dir,
    get_processed_data_dir, get_raw_data_dir, load_pickle, save_pickle,
)

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = False
PLOT_ONLY: bool = False
SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
SAVE_TYPES: tuple[str, ...] = ("pdf", "png", "eps")

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
BASE_RAW_PATH = RAW_DATA_DIR / "figure6_new_nk_landscapes.pkl"
STOCHASTIC_RAW_PATH = RAW_DATA_DIR / "figure6_i_modularity_gmu_global_local_raw.pkl"
ANALYTICAL_PATH = PROCESSED_DATA_DIR / "figure6_i_modularity_analytical_rho2.pkl"
FITTED_PATH = PROCESSED_DATA_DIR / "figure6_i_modularity_fitted_rho2.pkl"
print(f"PLOT_ONLY={PLOT_ONLY}, OVERWRITE_RAW_PKL={OVERWRITE_RAW_PKL}, OVERWRITE_PROCESSED_PKL={OVERWRITE_PROCESSED_PKL}")

## Raw Data Generation

In [ ]:
if not PLOT_ONLY and (OVERWRITE_RAW_PKL or not STOCHASTIC_RAW_PATH.exists()):
    command = [sys.executable, "scripts/generate_figure6_i_modularity_raw_data.py", "--raw-dir", str(RAW_DATA_DIR)]
    if OVERWRITE_RAW_PKL:
        command.append("--overwrite-raw")
    subprocess.run(command, check=True)
if not PLOT_ONLY and not STOCHASTIC_RAW_PATH.exists():
    raise FileNotFoundError(STOCHASTIC_RAW_PATH)
figure6_i_raw = None if PLOT_ONLY else load_pickle(STOCHASTIC_RAW_PATH)
if figure6_i_raw is not None:
    raw_data = figure6_i_raw["data"]
    assert np.asarray(raw_data["g_mu_global"]).shape == (4, 5, 20, 75)
    assert np.asarray(raw_data["g_mu_local"]).shape == (4, 5, 20, 20, 75)
    assert np.asarray(raw_data["local_start_coordinates"]).shape == (4, 20, 20, 4)
    assert bool(figure6_i_raw["params"]["local_starts_shared_across_lambda"])
    assert np.isfinite(np.asarray(raw_data["g_mu_global"])).all()
    assert np.isfinite(np.asarray(raw_data["g_mu_local"])).all()
    print("Validated full Figure 6I stochastic raw payload.")

## Processed Analytical $\rho_2$

In [ ]:
def process_analytical(base_payload: dict[str, object]) -> dict[str, object]:
    """Compute analytical rho_2 for every modular landscape.

    Parameters:
    - base_payload: dict[str, object]
        Paired base NK landscapes.

    Returns:
    - dict[str, object]
        Analytical replicate values and summaries.
    """
    landscapes = np.asarray(base_payload["data"]["landscapes"], dtype=np.float32)
    values = np.empty((len(K_VALUES), len(LAMBDA_VALUES), NUM_LANDSCAPES), dtype=float)
    scaling = np.empty(values.shape + (5,), dtype=float)
    for k_index in range(len(K_VALUES)):
        for lambda_index, lambda_value in enumerate(LAMBDA_VALUES):
            for landscape_index in range(NUM_LANDSCAPES):
                modified, stats = modular_landscape(landscapes[k_index, landscape_index], lambda_value)
                values[k_index, lambda_index, landscape_index] = float(get_dirichlet_metric(modified))
                scaling[k_index, lambda_index, landscape_index] = tuple(stats.values())
    return {"data": {"rho_2": values, "mean": values.mean(axis=2), "std": values.std(axis=2, ddof=1), "scaling_statistics": scaling},
            "params": {"N": N_SITES, "A": NUM_ALLELES, "K_values": K_VALUES, "lambda_values": LAMBDA_VALUES, "modules": MODULES, "num_landscapes": NUM_LANDSCAPES},
            "metadata": {"paper_reference": "Figure 6I", "standard_deviation_ddof": 1}}

if ANALYTICAL_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure6_i_analytical = load_pickle(ANALYTICAL_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(ANALYTICAL_PATH)
else:
    if not BASE_RAW_PATH.exists():
        raise FileNotFoundError(BASE_RAW_PATH)
    figure6_i_analytical = process_analytical(load_pickle(BASE_RAW_PATH))
    save_pickle(figure6_i_analytical, ANALYTICAL_PATH)
assert np.asarray(figure6_i_analytical["data"]["rho_2"]).shape == (4, 5, 20)

## Processed Global and Local Fitted $\rho_2$

In [ ]:
def process_fitted(raw_payload: dict[str, object]) -> dict[str, object]:
    """Fit global and one-start local rho_2 values.

    Parameters:
    - raw_payload: dict[str, object]
        Stochastic G_mu curves.

    Returns:
    - dict[str, object]
        Fits, summaries, and failures.
    """
    raw_data = raw_payload["data"]
    global_curves = np.asarray(raw_data["g_mu_global"], dtype=float)
    local_curves = np.asarray(raw_data["g_mu_local"], dtype=float)
    global_rho = np.full(global_curves.shape[:-1], np.nan)
    global_asymptote = np.full(global_curves.shape[:-1], np.nan)
    local_rho = np.full(local_curves.shape[:-1], np.nan)
    local_asymptote = np.full(local_curves.shape[:-1], np.nan)
    global_success = np.zeros(global_rho.shape, dtype=bool)
    local_success = np.zeros(local_rho.shape, dtype=bool)
    failures: list[dict[str, object]] = []
    mutation_scale = 2.0 * float(raw_payload["params"]["total_mutation_rate"])
    num_steps = int(raw_payload["params"]["num_decay_steps"])
    for fit_type, curves, rates, asymptotes, successes in (("global", global_curves, global_rho, global_asymptote, global_success), ("local", local_curves, local_rho, local_asymptote, local_success)):
        for index in np.ndindex(curves.shape[:-1]):
            try:
                curve = curves[index]
                if not np.all(np.isfinite(curve)) or np.isclose(curve[0], 0.0):
                    raise ValueError(f"{fit_type} G_mu is non-finite or begins at zero.")
                rate, asymptote = get_single_decay_rate(curve, mut=mutation_scale, num_steps=num_steps)
                rates[index], asymptotes[index], successes[index] = float(rate), float(asymptote), True
            except (RuntimeError, ValueError, FloatingPointError) as error:
                failures.append({"fit_type": fit_type, "index": tuple(int(value) for value in index), "error": str(error)})
    local_by_landscape = np.nanmean(local_rho, axis=3)
    return {"data": {"global_rho2": global_rho, "global_asymptote": global_asymptote, "global_fit_success": global_success,
                     "local_rho2": local_rho, "local_asymptote": local_asymptote, "local_fit_success": local_success,
                     "local_rho2_by_landscape": local_by_landscape, "global_mean": np.nanmean(global_rho, axis=2), "global_std": np.nanstd(global_rho, axis=2, ddof=1),
                     "local_mean": np.nanmean(local_by_landscape, axis=2), "local_std": np.nanstd(local_by_landscape, axis=2, ddof=1), "fit_failures": failures},
            "params": dict(raw_payload["params"]), "metadata": {"paper_reference": "Figure 6I", "num_fit_failures": len(failures), "standard_deviation_ddof": 1}}

if FITTED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure6_i_fitted = load_pickle(FITTED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(FITTED_PATH)
else:
    if figure6_i_raw is None:
        raise RuntimeError("Raw stochastic data are required for fitting.")
    figure6_i_fitted = process_fitted(figure6_i_raw)
    save_pickle(figure6_i_fitted, FITTED_PATH)
assert np.asarray(figure6_i_fitted["data"]["global_rho2"]).shape == (4, 5, 20)
assert np.asarray(figure6_i_fitted["data"]["local_rho2"]).shape == (4, 5, 20, 20)
print(f"Figure 6I fit failures: {figure6_i_fitted['metadata']['num_fit_failures']}")

## Summary and Figure 6I

In [ ]:
def save_figure(fig: Figure, stem: str) -> None:
    """Save a figure in configured formats.

    Parameters:
    - fig: Figure
        Figure to save.
    - stem: str
        Filename stem.

    Returns:
    - None
        Files are written below FIGURES_DIR.
    """
    for suffix in SAVE_TYPES:
        destination = FIGURES_DIR / suffix
        destination.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches="tight")

def plot_modularity(ax: Axes) -> None:
    """Plot analytical, global, and local Figure 6I estimates.

    Parameters:
    - ax: Axes
        Target axes.

    Returns:
    - None
        Artists are added in place.
    """
    analytical = figure6_i_analytical["data"]
    fitted = figure6_i_fitted["data"]
    lambdas = np.asarray(LAMBDA_VALUES)
    colors = ("tab:blue", "tab:orange", "tab:green", "tab:red")
    markers = ("o", "s", "D", "^")
    for index, k_value in enumerate(K_VALUES):
        for mean_key, std_key, style in (("mean", "std", "-"), ("global_mean", "global_std", "--"), ("local_mean", "local_std", "-.")):
            source = analytical if mean_key == "mean" else fitted
            mean, std = np.asarray(source[mean_key])[index], np.asarray(source[std_key])[index]
            ax.plot(lambdas, mean, color=colors[index], marker=markers[index], linestyle=style, linewidth=1.3, markersize=3.5)
            ax.fill_between(lambdas, mean - std, mean + std, color=colors[index], alpha=0.10, linewidth=0)
        ax.axhline((k_value + 1) / N_SITES, color=colors[index], linestyle=":", linewidth=1.0)
    k_handles = [mlines.Line2D([], [], color=colors[i], marker=markers[i], label=rf"$K={k}$") for i, k in enumerate(K_VALUES)]
    estimator_handles = [mlines.Line2D([], [], color="black", linestyle=style, label=label) for style, label in (("-", r"Analytical $\rho_2$"), ("--", r"$\rho_2^{\mathrm{glob}}$"), ("-.", r"$\rho_2^{\mathrm{loc}}$"), (":", r"$\rho_{NK}$"))]
    ax.legend(handles=k_handles + estimator_handles, ncol=2, fontsize=FIGURE_LEGEND_SIZE)
    ax.set_title(r"Increasing two-module structure ($N=4$, $A=20$)", fontsize=FIGURE_TITLE_SIZE)
    ax.set_xlabel(r"Modular projection strength, $\lambda$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(r"$\rho_2$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.grid(True, alpha=0.16)
    ax.text(-0.14, 1.10, "I", transform=ax.transAxes, fontsize=PANEL_LETTER_SIZE, fontweight="bold", va="top")

for k_index, k_value in enumerate(K_VALUES):
    print(f"K={k_value}: analytical={np.asarray(figure6_i_analytical['data']['mean'])[k_index]}, global={np.asarray(figure6_i_fitted['data']['global_mean'])[k_index]}, local={np.asarray(figure6_i_fitted['data']['local_mean'])[k_index]}")
fig, ax = plt.subplots(figsize=(4.2, 3.1), dpi=PANEL_DPI)
plot_modularity(ax)
if SAVE_FIGURES:
    save_figure(fig, "figure_6_I")
plt.show()